In [31]:
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report,average_precision_score
import optuna
from optuna.trial import Trial
from scipy.stats import ks_2samp

In [32]:
data = pd.read_parquet("/Users/prathikkundaragi/MY PROJECTS/Fraud Dataset/model_data_full.parquet")
data.head(5)

,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,9839.64,170136.0,160296.36,0.0,0.0,0,0,0,0,1,0
1,1864.28,21249.0,19384.72,0.0,0.0,0,0,0,0,1,0
2,181.00,181.0,0.00,0.0,0.0,1,0,0,0,0,1
3,181.00,181.0,0.00,21182.0,0.0,1,0,1,0,0,0
4,11668.14,41554.0,29885.86,0.0,0.0,0,0,0,0,1,0


In [33]:
# Split your `data` DataFrame into X and y 
X = data.drop(columns=["isFraud"]).values   # replace "isFraud" with your actual target column name
y = data["isFraud"].values

# Hold-out test split BEFORE tuning 

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,          # preserve fraud ratio in split
    random_state=42
)


In [34]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial,X,y):
    params = {
        "objective":        "binary:logistic",
        "eval_metric":      "aucpr",           # ← fraud scoring metric
        "tree_method":      "hist",
        "device":           "cpu",
        "random_state":     42,
        
        # --- params to tune ---
        "n_estimators":     trial.suggest_int("n_estimators", 100, 1000, step=50),
        "max_depth":        trial.suggest_int("max_depth", 3, 10),
        "learning_rate":    trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma":            trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        
        # critical for fraud imbalance
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 10, 150),
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucpr_scores, ks_scores = [], []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        model = xgb.XGBClassifier(**params, early_stopping_rounds=50, verbosity=0)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        probs = model.predict_proba(X_val)[:, 1]

        # Primary metric
        aucpr = average_precision_score(y_val, probs)
        aucpr_scores.append(aucpr)

        # Secondary: KS statistic
        fraud_scores = probs[y_val == 1]
        legit_scores = probs[y_val == 0]
        ks_stat, _   = ks_2samp(fraud_scores, legit_scores)
        ks_scores.append(ks_stat)

        # Report intermediate value for Optuna pruning
        trial.report(np.mean(aucpr_scores), step=fold)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    # Store KS as user attribute (for analysis later)
    trial.set_user_attr("ks_statistic", np.mean(ks_scores))

    return np.mean(aucpr_scores)   # ← Optuna maximizes this


In [35]:
# ── 4. Create and run the study ───────────────────────────────────────────
sampler = optuna.samplers.TPESampler(seed=42)
pruner  = optuna.pruners.MedianPruner(n_warmup_steps=2)

study = optuna.create_study(
    direction="maximize",
    sampler=sampler,
    pruner=pruner,
    study_name="xgb_fraud_scoring"
)

study.optimize(
    lambda trial: objective(trial, X_train, y_train),   
    n_trials=3,
    show_progress_bar=True
)

  0%|          | 0/3 [00:00<?, ?it/s]

In [36]:
best_trial = study.best_trial
print(f"Best AUC-PR  : {best_trial.value:.4f}")
print(f"Best KS Stat : {best_trial.user_attrs['ks_statistic']:.4f}")
print(f"Best Params  : {best_trial.params}")

# ── 6. Train final model on full X_train with best params ─────────────────
final_model = xgb.XGBClassifier(
    **study.best_params,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=42
)
final_model.fit(X_train, y_train)

Best AUC-PR  : 0.9279
Best KS Stat : 0.9882
Best Params  : {'n_estimators': 100, 'max_depth': 10, 'learning_rate': 0.11536162338241387, 'subsample': 0.6061695553391381, 'colsample_bytree': 0.5090949803242604, 'min_child_weight': 2, 'gamma': 1.5212112147976886, 'reg_alpha': 0.042051564509138696, 'reg_lambda': 0.01444525102276306, 'scale_pos_weight': 50.772079627725866}


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.5090949803242604, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='aucpr', feature_types=None, feature_weights=None,
              gamma=1.5212112147976886, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.11536162338241387,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=10, max_leaves=None,
              min_child_weight=2, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [38]:
test_probs   = final_model.predict_proba(X_test)[:, 1]
test_aucpr   = average_precision_score(y_test, test_probs)

fraud_s = test_probs[y_test == 1]
legit_s = test_probs[y_test == 0]
test_ks, _  = ks_2samp(fraud_s, legit_s)

print(f"Test AUC-PR  : {test_aucpr:.4f}")
print(f"Test KS Stat : {test_ks:.4f}")

Test AUC-PR  : 0.9338
Test KS Stat : 0.9906
